In [0]:

import re

from pyspark.sql.functions import col, current_timestamp


# =====================================================
# 1. JOB PARAMETERS
# =====================================================

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("target_schema", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("source_file_path", "")
dbutils.widgets.text("schema_location", "")
dbutils.widgets.text("checkpoint_location", "")
dbutils.widgets.text("table_comment", "")

catalog = dbutils.widgets.get("catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")
source_file_path = dbutils.widgets.get("source_file_path")
schema_location = dbutils.widgets.get("schema_location")
checkpoint_location = dbutils.widgets.get("checkpoint_location")
table_comment = dbutils.widgets.get("table_comment")


# =====================================================
# 2. VALIDATE PARAMETERS
# =====================================================

parameters = {
    "catalog": catalog,
    "target_schema": target_schema,
    "target_table": target_table,
    "source_file_path": source_file_path,
    "schema_location": schema_location,
    "checkpoint_location": checkpoint_location,
    "table_comment": table_comment
}

for name, value in parameters.items():

    if not value or not value.strip():
        raise ValueError(f"Missing required parameter: {name}")


# =====================================================
# 3. PREPARE SOURCE AND TARGET
# =====================================================

source_dir = source_file_path.rsplit("/", 1)[0]
file_name = source_file_path.rsplit("/", 1)[1]

target_table_fqn = (
    f"`{catalog}`.`{target_schema}`.`{target_table}`"
)

# Validate source file

files = dbutils.fs.ls(source_dir)

available_files = [f.name.rstrip("/") for f in files]

if file_name not in available_files:

    raise FileNotFoundError(
        f"Source file not found: {source_file_path}"
    )

# Create schema and checkpoint directories

dbutils.fs.mkdirs(schema_location)
dbutils.fs.mkdirs(checkpoint_location)


# =====================================================
# 4. READ JSON USING AUTO LOADER
# =====================================================

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("pathGlobFilter", file_name)
    .load(source_dir)
)


# =====================================================
# 5. ADD AUDIT COLUMNS
# =====================================================

df = (
    df
    .withColumn("load_dt", current_timestamp())
    .withColumn("source", col("_metadata.file_path"))
)


# =====================================================
# 6. STANDARDIZE COLUMN NAMES
# =====================================================

def clean_column_names(df):

    cleaned_columns = []
    column_counts = {}

    for column in df.columns:

        clean_name = re.sub(
            r"[^a-zA-Z0-9_]+",
            "_",
            column.strip()
        )

        clean_name = re.sub(r"_+", "_", clean_name)
        clean_name = clean_name.strip("_").lower()

        if not clean_name:
            clean_name = "column"

        column_counts[clean_name] = (
            column_counts.get(clean_name, 0) + 1
        )

        if column_counts[clean_name] > 1:
            clean_name = (
                f"{clean_name}_{column_counts[clean_name]}"
            )

        cleaned_columns.append(clean_name)

    return df.toDF(*cleaned_columns)


df = clean_column_names(df)

print("Final Bronze Schema:")
df.printSchema()


# =====================================================
# 7. WRITE DATA INTO BRONZE DELTA TABLE
# =====================================================

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(target_table_fqn)
)

query.awaitTermination()


# =====================================================
# 8. ADD TABLE AND COLUMN METADATA
# =====================================================

safe_comment = table_comment.replace("'", "''")

spark.sql(f"""
    COMMENT ON TABLE {target_table_fqn}
    IS '{safe_comment}'
""")

spark.sql(f"""
    ALTER TABLE {target_table_fqn}
    ALTER COLUMN load_dt
    COMMENT 'Timestamp when the record was ingested into Bronze'
""")

spark.sql(f"""
    ALTER TABLE {target_table_fqn}
    ALTER COLUMN source
    COMMENT 'Source file path from which the record was ingested'
""")


# =====================================================
# 9. FINAL VALIDATION
# =====================================================

record_count = spark.table(target_table_fqn).count()

print("-----------------------------------")
print("BRONZE INGESTION COMPLETED")
print("-----------------------------------")
print(f"Target Table : {target_table_fqn}")
print(f"Source File  : {source_file_path}")
print(f"Record Count : {record_count}")
print("-----------------------------------")
